In [0]:
select count(*)
from kagr_hbse.stage.vw_sixers_seatmapgamesummary_enhanced
-- where PURCHASEPRICE <= 0 AND RESALEATP <=0
where saledate >= '2023-07-01 00:00:00.000';

select count(*)
from hbse.default.sixers_rfm_dataset;
/************************ Dataset (game summary full) *******************************************/
CREATE or REPLACE TABLE hbse.default.sixers_rfm_dataset AS
WITH base_data AS (
  SELECT
     d.audienceid, to_date(c.eventdate) as saledate,
      c.ledgername, count(c.seatnumber) as total_seats, sum(c.purchaseprice) as primary_cost, sum(c.resaleatp) as secondary_cost, (COALESCE(SUM(c.purchaseprice), 0) + COALESCE(SUM(c.resaleatp), 0)) as total_cost
  FROM
      kagr_hbse.stage.vw_sixers_seatmapgamesummary_enhanced c
      join kagr_sixers.stage.audiencemapping d on c.rawaudienceid = d.rawaudienceid
WHERE saledate >= '2023-07-01 00:00:00.000'
GROUP BY d.audienceid, to_date(c.eventdate), c.ledgername
)
SELECT
  audienceid, saledate, ledgername,
  SUM(total_seats) AS total_seats,
  CAST(SUM(total_cost) AS DOUBLE) AS total_sales
FROM base_data
GROUP BY audienceid, saledate, ledgername;
